# 05 — Generative LLM Evaluation (single-prompt strategy)

Evaluates three generative LLMs on the **exact v2 test split (2,656 clauses)**,
**same scoring code** (`scripts/lawgic_eval_core.py`) and **same metrics** as the
encoder runs in `02_multiseed_encoder_runs.ipynb`, so the two model families are
directly comparable.

**This is a "test the waters" run.** Goal: prove the end-to-end pipeline works
(prompt → LLM → parse → pseudo-logits → identical scoring → metrics), not to
produce final numbers. Run the **10-clause smoke batch per model first**
(`SMOKE_TEST = True`, cell "Smoke runner"), eyeball it, then flip
`SMOKE_TEST = False` for the full 2,656-clause run.

Models (this pass): `gemma4:31b-cloud` (Ollama Cloud, via the local daemon) and
`adrienbrault/saul-instruct-v1:Q8_0` (local, M1). `qwen3.5:cloud` is deferred — it
returns HTTP 402 (needs Ollama cloud credits); re-add it in the config once billing
is enabled. Decoding is greedy (`temperature=0.0`), one pass per clause (§4.4).

Scoring integration: an LLM returns a *decision*, not a score, so we build
**pseudo-logits** (`+10` for a predicted class, `-10` otherwise) that reproduce
the same decision after the scoring module's `sigmoid`/`argmax` — no scoring code
changes.


## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

# ── Corpus version: MUST be set BEFORE importing lawgic_eval_core ─────────────
os.environ["LAWGIC_CORPUS_VERSION"] = "v2"


def find_project_root(start: Path) -> Path:
    sentinel = "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv"
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv

import lawgic_eval_core as core

load_dotenv(PROJECT_ROOT / ".env")
OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")

# Corpus + split + label arrays (identical to the encoder notebook).
core.persist_split()
corpus = core.load_corpus()
test = core.split_frames(corpus)["test"].reset_index(drop=True)
arrays = core.label_arrays(test)

TOPIC_IDS = core.TOPIC_IDS
NAME_BY_ID = core.TOPIC_NAME_BY_ID
TAXONOMY = json.loads(core.TAXONOMY_PATH.read_text(encoding="utf-8"))

assert len(TOPIC_IDS) == 42, len(TOPIC_IDS)
assert len(test) == 2656, len(test)
assert core.HARM_CLASS_NAMES == {0: "Harmful", 1: "Neutral", 2: "Fair"}

print(f"Corpus version : {core._CORPUS_VERSION}")
print(f"Topics         : {len(TOPIC_IDS)}")
print(f"Test clauses   : {len(test)}")
print(f"OLLAMA_API_KEY : {'set' if OLLAMA_API_KEY else 'not set'} (unused — cloud runs via the local `ollama` daemon / `ollama signin`)")
print(f"Eval out dir   : {core.EVAL_OUT_DIR}")


Corpus version : v2
Topics         : 42
Test clauses   : 2656
OLLAMA_API_KEY : set (unused — cloud runs via the local `ollama` daemon / `ollama signin`)
Eval out dir   : /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2


## 2. Config — flags, models, dirs, risk map

In [2]:
# ── TEST-THE-WATERS SWITCH ───────────────────────────────────────────────────
SMOKE_TEST = False      # True: 10-clause stratified smoke per model (writes _smoke/)
N_SMOKE = 10           # --- FLIP SMOKE_TEST TO False FOR THE FULL RUN (2,656 clauses) ---
SMOKE_SEED = 42        # fixed so the same 10 clauses are reused across models/reruns

# One runner, three configs. ALL run through the local `ollama` daemon
# (http://localhost:11434): the `:cloud` tags proxy to Ollama Cloud via your
# `ollama signin` (no API key); the Saul tag runs on-device. `location` is kept
# only as a report label. NOTE: cloud models still hit Ollama's per-account gates
# — e.g. `qwen3.5:cloud` returns HTTP 402 (needs cloud credits) until billing is
# enabled; that failure is recorded and retried on the next run (see runner).
MODELS = [
    # qwen3.5:cloud dropped this pass — HTTP 402 (needs Ollama cloud credits).
    # Uncomment to re-add once billing is enabled at https://ollama.com/settings:
    # {"run_id": "qwen3.5-cloud__single",     "model": "qwen3.5:cloud",       "location": "cloud"},
    {"run_id": "gemma4-31b-cloud__single",    "model": "gemma4:31b-cloud",    "location": "cloud"},
    {"run_id": "saul-instruct-v1-q8__single", "model": "adrienbrault/saul-instruct-v1:Q8_0", "location": "local"},
]

# Ollama silently truncates past num_ctx, so it must clear the prompt: the rendered
# prompt measures ~6.5k tokens (full 42-topic taxonomy — larger than the plan's ~4k
# estimate), so num_ctx=12288 leaves headroom for prompt + JSON. Still modest for a
# 16 GB M1 (well under the 32768 that would blow the KV cache). See preflight.
NUM_CTX = 12288

# Cap generation length. Without it a small quantised model (Saul) can loop into a
# multi-minute repetition runaway — one smoke clause spat thousands of garbled topic
# strings over 725 s. A valid answer (≤~15 topic ids + risk) is far under 768 tokens;
# this also bounds worst-case latency for the full 2,656-clause pass.
NUM_PREDICT = 768

# Artifact dirs (mirror Phase 2). Full runs live directly under generative_runs/;
# smoke runs go to a separate _smoke/ subdir so they never mix with the full run.
BASE_DIR = core.EVAL_OUT_DIR / "generative_runs"
SMOKE_DIR = BASE_DIR / "_smoke"

# ── Risk label mapping — centralised, easy to get backwards ───────────────────
# Encoder ground truth harm_class: 0=Harmful, 1=Neutral, 2=Fair (core.HARM_CLASS_NAMES).
# LLM output vocabulary matches encoder naming.
RISK_TO_CLASS = {"harmful": 0, "neutral": 1, "fair": 2}
DEFAULT_RISK_ON_FAILURE = "neutral"   # used only if a call fails to parse (counted)
assert {v: k.capitalize() for k, v in RISK_TO_CLASS.items()} == core.HARM_CLASS_NAMES

# Pseudo-logit magnitudes: sigmoid(+10)≈1≥0.5, sigmoid(-10)≈0; argmax recovers class.
POS_LOGIT, NEG_LOGIT = 10.0, -10.0


## 3. Prompt, output schema, parse & pseudo-logit helpers

In [3]:
import re
from typing import Literal

from pydantic import BaseModel, Field


class ClausePrediction(BaseModel):
    topics: list[str] = Field(default_factory=list,
                              description="Applicable topic IDs from the taxonomy")
    risk: Literal["harmful", "neutral", "fair"] = Field(
        description="One overall risk class for the whole clause")


def render_taxonomy_block(taxonomy: dict, topic_ids: list[str]) -> str:
    """Inline id/name/description + 3-level risk rubric for each of the 42 topics."""
    by_id = {t["id"]: t for t in taxonomy["topics"]}
    lines = []
    for tid in topic_ids:
        t = by_id[tid]
        lines.append(f"- id: {tid}")
        lines.append(f"  name: {t.get('name', tid)}")
        lines.append(f"  description: {(t.get('description') or '').strip()}")
        lines.append("  risk_rubric:")
        for s in t.get("scores", []):
            lines.append(f"    - {s['label']}: {s['explanation'].strip()}")
    return "\n".join(lines)


TAXONOMY_BLOCK = render_taxonomy_block(TAXONOMY, TOPIC_IDS)

INSTRUCTION = (
    "You are a strict classifier of Terms-of-Service (ToS) clauses.\n"
    f"Below is a taxonomy of {len(TOPIC_IDS)} topics, each with a name, description "
    "and a 3-level risk rubric.\n"
    "Given the clause, return (1) every applicable topic ID from the taxonomy, and "
    "(2) a SINGLE overall risk class for the whole clause: exactly one of "
    "harmful, neutral, fair.\n"
    "Use only topic IDs that appear in the taxonomy. Return strict JSON only.\n"
)


def build_prompt(clause_text: str) -> str:
    return (
        INSTRUCTION
        + "\nTAXONOMY:\n" + TAXONOMY_BLOCK
        + "\n\nCLAUSE:\n" + str(clause_text).strip()
        + '\n\nReturn JSON only: {"topics": ["<id>", ...], "risk": "harmful|neutral|fair"}'
    )


# ── Topic ID validation: accept exact id OR normalized name; count unknowns ───
def _norm(s: str) -> str:
    return re.sub(r"[\s_]+", " ", str(s).strip().lower())


def build_topic_lookup(topic_ids: list[str], name_by_id: dict) -> dict:
    lut = {}
    for idx, tid in enumerate(topic_ids):
        lut[_norm(tid)] = idx
        nm = name_by_id.get(tid)
        if nm:
            lut[_norm(nm)] = idx
    return lut


TOPIC_LOOKUP = build_topic_lookup(TOPIC_IDS, NAME_BY_ID)


def map_topics(pred_topics) -> tuple[list[int], list[str]]:
    """Return (column indices, dropped labels) — drops unknown ids/names, no crash."""
    idxs, dropped = [], []
    for t in (pred_topics or []):
        i = TOPIC_LOOKUP.get(_norm(t))
        (idxs.append(i) if i is not None else dropped.append(t))
    return sorted(set(idxs)), dropped


# ── Pseudo-logit builders ─────────────────────────────────────────────────────
def topic_logits_row(idxs: list[int]) -> np.ndarray:
    row = np.full(len(TOPIC_IDS), NEG_LOGIT, dtype=np.float32)
    if idxs:
        row[idxs] = POS_LOGIT
    return row


def harm_logits_row(risk: str) -> np.ndarray:
    row = np.full(3, NEG_LOGIT, dtype=np.float32)
    row[RISK_TO_CLASS[risk if risk in RISK_TO_CLASS else DEFAULT_RISK_ON_FAILURE]] = POS_LOGIT
    return row


## 4. Model builder + per-clause predict

In [4]:
from langchain_ollama import ChatOllama


def build_llm(cfg: dict) -> ChatOllama:
    """One code path for all three, through the local daemon. `:cloud` tags proxy
    to Ollama Cloud via `ollama signin` (no API key); local tags run on-device.
    num_predict caps generation so a small model can't loop into a runaway."""
    return ChatOllama(
        model=cfg["model"], base_url="http://localhost:11434",
        temperature=0.0, format="json", num_ctx=NUM_CTX, num_predict=NUM_PREDICT,
    )


def predict_clause(structured_llm, clause_text: str) -> dict:
    """One greedy pass. Never crashes: parse failure -> parse_ok=False, neutral fallback."""
    t0 = time.perf_counter()
    topics, risk, raw, parse_ok = [], DEFAULT_RISK_ON_FAILURE, "", True
    try:
        out = structured_llm.invoke(build_prompt(clause_text))
        topics, risk, raw = list(out.topics), out.risk, out.model_dump_json()
    except Exception as exc:  # invalid JSON, schema violation, transport error
        parse_ok, raw = False, f"PARSE_ERROR: {exc}"[:2000]
    return {
        "parsed_topics": topics, "parsed_risk": risk, "raw_json": raw,
        "parse_ok": parse_ok, "latency_s": round(time.perf_counter() - t0, 3),
    }


## 5. Runner (resumable) + assembler + smoke sampler

In [5]:
def _raw_csv_path(cfg: dict, base_dir: Path) -> Path:
    return base_dir / cfg["run_id"] / "raw_predictions.csv"


def _read_raw(raw_csv: Path) -> pd.DataFrame:
    """Read raw_predictions.csv, normalize parse_ok to bool, and keep the LAST
    attempt per row_id (a retried row is appended, so the newest wins)."""
    df = pd.read_csv(raw_csv)
    df["row_id"] = df["row_id"].astype(int)
    df["parse_ok"] = df["parse_ok"].map(lambda x: str(x).strip().lower() in ("true", "1"))
    return df.drop_duplicates("row_id", keep="last")


def _completed_row_ids(raw_csv: Path) -> set:
    """Row_ids that SUCCEEDED. Failures (parse_ok=False: 402/401/timeout/bad JSON)
    are deliberately excluded so the next run retries them."""
    if not raw_csv.exists():
        return set()
    df = _read_raw(raw_csv)
    return set(df.loc[df["parse_ok"], "row_id"])


def run_model(cfg: dict, positions, base_dir: Path, verbose: bool = False):
    """Run cfg over the given positional indices into `test`. Resumable: appends
    each clause to raw_predictions.csv as it arrives and skips already-done row_ids.
    Returns (raw_csv_path, wall_seconds)."""
    raw_csv = _raw_csv_path(cfg, base_dir)
    raw_csv.parent.mkdir(parents=True, exist_ok=True)
    done = _completed_row_ids(raw_csv)

    structured = build_llm(cfg).with_structured_output(ClausePrediction)
    wall = 0.0
    for n, p in enumerate(positions, start=1):
        rid = int(test.iloc[p]["row_id"])
        if rid in done:
            continue
        rec = predict_clause(structured, test.iloc[p]["text"])
        wall += rec["latency_s"]
        row = {
            "row_id": rid, "raw_json": rec["raw_json"],
            "parsed_topics": json.dumps(rec["parsed_topics"]),
            "parsed_risk": rec["parsed_risk"], "parse_ok": rec["parse_ok"],
            "latency_s": rec["latency_s"],
        }
        header = not raw_csv.exists()
        pd.DataFrame([row]).to_csv(raw_csv, mode="a", header=header, index=False)
        done.add(rid)
        if verbose:
            print(f"  [{n}/{len(positions)}] row_id={rid} risk={rec['parsed_risk']} "
                  f"topics={rec['parsed_topics']} ok={rec['parse_ok']} {rec['latency_s']}s")
    return raw_csv, wall


def assemble_logits(raw: pd.DataFrame, frame: pd.DataFrame):
    """Build (topic_logits, harm_logits, dropped_count, parse_failures) in `frame`
    row order from a raw_predictions dataframe keyed by row_id."""
    by_id = raw.set_index("row_id")   # raw is deduped by _read_raw -> unique index
    tl, hl, dropped, failures = [], [], 0, 0
    for rid in frame["row_id"].astype(int):
        rec = by_id.loc[rid]
        topics = rec["parsed_topics"]
        topics = json.loads(topics) if isinstance(topics, str) else list(topics or [])
        idxs, drops = map_topics(topics)
        dropped += len(drops)
        failures += int(not bool(rec["parse_ok"]))   # parse_ok is real bool via _read_raw
        tl.append(topic_logits_row(idxs))
        hl.append(harm_logits_row(rec["parsed_risk"]))
    return np.vstack(tl), np.vstack(hl), dropped, failures


def stratified_smoke_positions(n: int, seed: int) -> list[int]:
    """Positional indices into `test`: one clause per observed risk class, then
    random fill, all over harm-observed rows so risk mini-metrics are meaningful."""
    rng = np.random.default_rng(seed)
    harm, hmask = arrays["harm_labels"], arrays["harm_masks"].astype(bool)
    picks = []
    for cls in (0, 1, 2):
        cand = np.where((harm == cls) & hmask)[0]
        if len(cand):
            picks.append(int(rng.choice(cand)))
    pool = np.where(hmask)[0].tolist()
    rng.shuffle(pool)
    for p in pool:
        if len(picks) >= n:
            break
        if p not in picks:
            picks.append(p)
    return picks[:n]


## 6. Preflight checks

In [6]:
# ── Offline structural self-checks (no LLM calls) ────────────────────────────
# ponytail: this is the runnable check for the pseudo-logit + risk-map + parse logic.

# 1) Pseudo-logit round-trip: feeding the gold labels back through the scoring
#    module must yield ~1.0 on observed cells (proves the +10/-10 trick + column
#    order + risk map are correct end to end).
_gold_tl = np.where(arrays["labels"] > 0.5, POS_LOGIT, NEG_LOGIT).astype(np.float32)
_gold_hl = np.full((len(test), 3), NEG_LOGIT, dtype=np.float32)
for _i, _c in enumerate(arrays["harm_labels"]):
    _gold_hl[_i, _c if _c in (0, 1, 2) else 1] = POS_LOGIT
_m = core.all_metrics(_gold_tl, _gold_hl, arrays)
assert _m["topic_micro_f1"] > 0.999 and _m["risk_accuracy"] > 0.999, _m
print(f"round-trip: gold->pseudo-logits->scoring  topic_micro_f1={_m['topic_micro_f1']:.3f} "
      f"risk_acc={_m['risk_accuracy']:.3f} ✓")

# 2) Risk map matches encoder ground truth.
assert RISK_TO_CLASS == {"harmful": 0, "neutral": 1, "fair": 2}
assert {v: k.capitalize() for k, v in RISK_TO_CLASS.items()} == core.HARM_CLASS_NAMES
print("risk map: harmful->0, neutral->1, fair->2 matches core.HARM_CLASS_NAMES ✓")

# 3) Topic lookup accepts id AND name; unknowns are dropped (not crashed).
_idxs, _drops = map_topics([TOPIC_IDS[0], NAME_BY_ID[TOPIC_IDS[3]], "not_a_real_topic"])
assert _idxs == sorted({0, 3}) and _drops == ["not_a_real_topic"], (_idxs, _drops)
print(f"topic lookup: id+name resolve, unknown dropped (idxs={_idxs}, dropped={_drops}) ✓")

# 4) Prompt token budget vs num_ctx (chars/4 estimate; assert well under NUM_CTX).
_sample_prompt = build_prompt(test.iloc[0]["text"])
_est_tokens = len(_sample_prompt) // 4
print(f"rendered prompt: {len(_sample_prompt):,} chars  ~{_est_tokens:,} tokens (est.)  "
      f"num_ctx={NUM_CTX}")
assert _est_tokens < NUM_CTX * 0.9, f"prompt ~{_est_tokens} tokens too close to num_ctx={NUM_CTX}"


round-trip: gold->pseudo-logits->scoring  topic_micro_f1=1.000 risk_acc=1.000 ✓
risk map: harmful->0, neutral->1, fair->2 matches core.HARM_CLASS_NAMES ✓
topic lookup: id+name resolve, unknown dropped (idxs=[0, 3], dropped=['not_a_real_topic']) ✓
rendered prompt: 26,122 chars  ~6,530 tokens (est.)  num_ctx=12288


In [7]:
# ── Live preflight (needs a running model) — run before a full pass ──────────
# Confirms Ollama is reachable and, critically for local Saul, that the prompt
# is NOT silently truncated (prompt_eval_count < num_ctx). Ollama truncates
# overflow with no error, which would make Saul score garbage.
def live_preflight(cfg: dict):
    llm = build_llm(cfg)
    resp = llm.invoke(build_prompt(test.iloc[0]["text"]))
    meta = getattr(resp, "response_metadata", {}) or {}
    usage = getattr(resp, "usage_metadata", {}) or {}
    pec = meta.get("prompt_eval_count") or usage.get("input_tokens")
    print(f"{cfg['run_id']}: prompt_eval_count={pec} num_ctx={NUM_CTX}")
    if pec is not None:
        assert pec < NUM_CTX, f"TRUNCATION: prompt_eval_count {pec} >= num_ctx {NUM_CTX}"
    return pec

# SaulLM local prerequisites: `ollama serve` running, model pulled
# (`ollama pull adrienbrault/saul-instruct-v1:Q8_0`), and heavy apps closed
# (Q8_0 is 7.7 GB; 16 GB M1 fits it with num_ctx=8192). Quit browsers first.
#
# Uncomment to run when a model is up and (for cloud) OLLAMA_API_KEY is set:
# for _cfg in MODELS:
#     try:
#         live_preflight(_cfg)
#     except Exception as _e:
#         print(f"{_cfg['run_id']}: not ready -> {_e}")


## 7. Smoke runner (10 clauses/model) — RUN FIRST

In [8]:
# Smoke: 10 stratified clauses/model, printed for eyeballing. Writes to _smoke/.
# ← RUN THIS FIRST. Inspect the raw JSON, parsed prediction and mini-metrics,
#   fix any bugs, THEN set SMOKE_TEST = False and run the full runner cell.
if SMOKE_TEST:
    smoke_pos = stratified_smoke_positions(N_SMOKE, SMOKE_SEED)
    print(f"Smoke positions (n={len(smoke_pos)}): {smoke_pos}")
    print("Gold risk classes:", [core.HARM_CLASS_NAMES.get(int(c), "n/a")
                                  for c in arrays["harm_labels"][smoke_pos]])
    print("=" * 100)
    for cfg in MODELS:
        print(f"\n### {cfg['run_id']}  ({cfg['model']}, {cfg['location']})")
        try:
            raw_csv, wall = run_model(cfg, smoke_pos, SMOKE_DIR, verbose=True)
        except Exception as exc:
            print(f"  SKIPPED — {exc}")
            continue
        raw = _read_raw(raw_csv)
        raw = raw[raw["row_id"].isin(test.iloc[smoke_pos]["row_id"])]
        tl, hl, dropped, failures = assemble_logits(raw, test.iloc[smoke_pos])
        mini = core.all_metrics(tl, hl, {k: v[smoke_pos] for k, v in arrays.items()})
        print(f"  mini-metrics: {{'topic_macro_f1': {mini['topic_macro_f1']:.3f}, "
              f"'risk_accuracy': {mini['risk_accuracy']:.3f}}}  "
              f"dropped_labels={dropped} parse_failures={failures} wall={wall:.1f}s")
    print("\n" + "=" * 100)
    print("Smoke done. Inspect above. When happy: set SMOKE_TEST = False, re-run"
          " config + this section's cells, then run the FULL runner cell.")
else:
    print("SMOKE_TEST = False -> skipping smoke. Run the full runner cell.")


SMOKE_TEST = False -> skipping smoke. Run the full runner cell.


## 8. Full runner (2,656 clauses) — set SMOKE_TEST=False

TODO: have separate cells and stuf for gemma and saul so it doesnt rely on one loop...

In [9]:
# --- FLIP SMOKE_TEST TO False (config cell) FOR THE FULL RUN (2,656 clauses) ---
# Resumable: interrupt any time (cloud quota / kernel restart) and re-run — it
# skips row_ids already in each model's raw_predictions.csv. Writes to
# generative_runs/<run_id>/. Cloud models have daily limits; running one model
# per day is fine — progress is never lost.
if not SMOKE_TEST:
    all_positions = list(range(len(test)))
    for cfg in MODELS:
        print(f"\n### {cfg['run_id']}  ({cfg['model']}, {cfg['location']})")
        raw_csv = _raw_csv_path(cfg, BASE_DIR)
        already = len(_completed_row_ids(raw_csv))
        print(f"  resuming: {already}/{len(test)} already done")
        try:
            _, wall = run_model(cfg, all_positions, BASE_DIR, verbose=False)
            print(f"  finished this pass in {wall / 60:.1f} min "
                  f"({len(_completed_row_ids(raw_csv))}/{len(test)} total)")
        except Exception as exc:
            print(f"  STOPPED — {exc}. Re-run to resume from where it left off.")
else:
    print("SMOKE_TEST = True -> full runner skipped. Flip to False to run 2,656 clauses.")



### gemma4-31b-cloud__single  (gemma4:31b-cloud, cloud)
  resuming: 0/2656 already done
  finished this pass in 33.9 min (1500/2656 total)

### saul-instruct-v1-q8__single  (adrienbrault/saul-instruct-v1:Q8_0, local)
  resuming: 0/2656 already done
  finished this pass in 459.5 min (2656/2656 total)


## 9. Aggregation → metrics.json / predictions.npz / headline

In [10]:
# Assemble pseudo-logits per model -> core.all_metrics -> artifacts. Operates on
# the FULL runs under generative_runs/. Safe to run anytime; skips models whose
# run is incomplete.
def aggregate_model(cfg: dict) -> dict | None:
    raw_csv = _raw_csv_path(cfg, BASE_DIR)
    if not raw_csv.exists():
        print(f"{cfg['run_id']}: no raw_predictions.csv yet — skipped.")
        return None
    raw = _read_raw(raw_csv)
    have = set(raw["row_id"])
    need = set(test["row_id"].astype(int))
    if not need <= have:
        print(f"{cfg['run_id']}: incomplete ({len(have & need)}/{len(need)} attempted) — skipped.")
        return None
    n_ok = int(raw["parse_ok"].sum())
    if n_ok < len(need):
        print(f"{cfg['run_id']}: WARNING {len(need) - n_ok} rows failed to parse "
              f"(e.g. 402/401/timeout) — scored as empty/neutral; see parse_failures.")

    tl, hl, dropped, failures = assemble_logits(raw, test)
    metrics = core.all_metrics(tl, hl, arrays)
    wall = float(raw["latency_s"].sum())

    out_dir = BASE_DIR / cfg["run_id"]
    np.savez_compressed(
        out_dir / "predictions.npz",
        topic_logits=tl, harm_logits=hl, row_id=test["row_id"].to_numpy(),
        # full array bundle too, so the encoder notebook's load_run_logits/compare
        # (for the deferred significance TODO) works verbatim.
        labels=arrays["labels"], label_masks=arrays["label_masks"],
        harm_labels=arrays["harm_labels"], harm_masks=arrays["harm_masks"],
    )
    payload = {
        "run_id": cfg["run_id"], "model": cfg["model"], "location": cfg["location"],
        "strategy": "single-prompt", "corpus_version": "v2", "num_ctx": NUM_CTX,
        "n_clauses": int(len(test)), "wall_seconds": wall,
        "latency_s_mean": float(raw["latency_s"].mean()),
        "dropped_label_count": int(dropped), "parse_failures": int(failures),
        **{m: metrics[m] for m in core.HEADLINE_METRICS},
        **{k: metrics[k] for k in ("topic_weighted_f1", "risk_weighted_f1", "rows")},
    }
    (out_dir / "metrics.json").write_text(json.dumps(payload, indent=2))
    print(f"{cfg['run_id']}: saved metrics.json + predictions.npz  "
          f"(dropped={dropped}, parse_failures={failures}, wall={wall / 60:.1f}min)")
    return payload


records = [r for r in (aggregate_model(cfg) for cfg in MODELS) if r]
if records:
    headline = pd.DataFrame([
        {"Model": r["run_id"],
         **{core.HEADLINE_METRICS[i]: r[core.HEADLINE_METRICS[i]] for i in range(len(core.HEADLINE_METRICS))},
         "dropped_labels": r["dropped_label_count"], "parse_failures": r["parse_failures"],
         "wall_min": round(r["wall_seconds"] / 60, 1)}
        for r in records
    ])
    display(headline)
    csv_path, tex_path = core.write_outputs(
        headline, "generative_headline",
        "Generative LLM single-prompt evaluation on the v2 test split (2,656 clauses).",
        "tab:generative_headline")
    print(f"Wrote {csv_path.name} + {tex_path.name} to {core.EVAL_OUT_DIR}")
else:
    print("No completed model runs yet. Run the full runner first.")


gemma4-31b-cloud__single: WARNING 1156 rows failed to parse (e.g. 402/401/timeout) — scored as empty/neutral; see parse_failures.
gemma4-31b-cloud__single: saved metrics.json + predictions.npz  (dropped=0, parse_failures=1156, wall=33.9min)
saul-instruct-v1-q8__single: saved metrics.json + predictions.npz  (dropped=7440, parse_failures=0, wall=459.5min)


,Model,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1,dropped_labels,parse_failures,wall_min
0,gemma4-31b-cloud__single,0.306257,0.322427,0.472515,0.437081,0,1156,33.9
1,saul-instruct-v1-q8__single,0.135815,0.145155,0.469127,0.212882,7440,0,459.5


Wrote generative_headline.csv + generative_headline.tex to /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2


## 10. Per-topic tables + inference cost

In [11]:
# Per-topic P/R/F1/support per model.
for cfg in MODELS:
    npz = BASE_DIR / cfg["run_id"] / "predictions.npz"
    if not npz.exists():
        continue
    payload = np.load(npz)
    table = core.per_topic_table(payload["topic_logits"], arrays, core.TOPIC_IDS)
    table.to_csv(BASE_DIR / cfg["run_id"] / "per_topic.csv", index=False)
    print(f"{cfg['run_id']}: per_topic.csv ({len(table)} rows) — macro/weighted tail:")
    display(table.tail(2))


gemma4-31b-cloud__single: per_topic.csv (44 rows) — macro/weighted tail:


,topic_id,precision,recall,f1,support,observed
42,macro avg,0.53181,0.268872,0.306257,4725,91863
43,weighted avg,0.60424,0.229418,0.283303,4725,91863


saul-instruct-v1-q8__single: per_topic.csv (44 rows) — macro/weighted tail:


,topic_id,precision,recall,f1,support,observed
42,macro avg,0.085265,0.538698,0.135815,4725,91863
43,weighted avg,0.118477,0.484233,0.179645,4725,91863


In [12]:
# Inference-cost logging (§4.4: report cost alongside accuracy). Local Saul is
# far slower than cloud — expected, and recorded.
cost_rows = []
for cfg in MODELS:
    mp = BASE_DIR / cfg["run_id"] / "metrics.json"
    if not mp.exists():
        continue
    m = json.loads(mp.read_text())
    cost_rows.append({"Model": m["run_id"], "location": m["location"],
                      "n_clauses": m["n_clauses"], "latency_s_mean": round(m["latency_s_mean"], 2),
                      "wall_min": round(m["wall_seconds"] / 60, 1)})
if cost_rows:
    display(pd.DataFrame(cost_rows))
else:
    print("No completed runs to cost yet.")


,Model,location,n_clauses,latency_s_mean,wall_min
0,gemma4-31b-cloud__single,cloud,2656,0.77,33.9
1,saul-instruct-v1-q8__single,local,2656,10.38,459.5


## TODO — deferred to the later full comparison

- **Generative-vs-encoder significance** (McNemar + paired bootstrap vs the best
  Legal-BERT dual run from Phase 2). Deferred per the user's decision for this
  "test the waters" pass. When adding it: load an encoder run's `test_logits.npz`
  and each model's `predictions.npz`, **assert
  `np.array_equal(row_id_generative, row_id_encoder)`** before any paired test
  (copy the `compare()` guard from `02_multiseed_encoder_runs.ipynb`). The npz
  files here already bundle the full label/mask arrays so `compare()` works verbatim.
- **Other strategies**: two-stage and retrieval-augmented (out of scope here).
- **Manuscript number sync**: the `.tex` still says 44 topics / 2,648 test in
  places; target is v2 = 42 topics / 2,656 test.
- **SaulLM fidelity caveat**: quantized Q8_0; its bundled Ollama template is
  ChatML (`<|im_start|>`), not the Mistral `[INST]` its HF card documents. The
  chat API applies the template server-side, so `ChatOllama` needs no manual
  templating, but record the quant + template as a fidelity caveat.
